# IZQAR Unified Intelligence — four-task validation experiment

Attach four inputs before running: `ealaxi/paysim1`, `ieee-fraud-detection` (after accepting its competition rules yourself), `sgpjesus/bank-account-fraud-dataset-neurips-2022`, and `ealtman2019/ibm-transactions-for-anti-money-laundering-aml`.

This performs genuine joint optimisation: four feature adapters, one shared backbone and four independent heads. Labels are not combined. It only uses each source's earlier train period and validation period; the existing final test windows are sealed. The result is experimental until independently compared with the specialised models.

In [ ]:
from pathlib import Path
import subprocess, sys
root = Path('/kaggle/working/izqar_unified')
if root.exists():
    raise FileExistsError('Use a fresh Kaggle session; refusing a stale checkout')
subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'feat/ml-pipelines-and-startup', 'https://github.com/Night011/izqar.git', str(root)], check=True)
backend = root / 'backend'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-r', str(backend / 'requirements-ml.txt')], check=True)
# Kaggle Python images normally include torch. The command makes a missing dependency explicit.
import torch
print('torch', torch.__version__, 'CUDA', torch.cuda.is_available())
print('Code commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=root, text=True).strip())


In [ ]:
inputs = Path('/kaggle/input')
def exactly_one(filename, marker):
    found = [p for p in inputs.rglob(filename) if marker in str(p)]
    if len(found) != 1: raise FileNotFoundError(f'Expected one {filename} from {marker}; found {found}')
    return found[0]
paysim = exactly_one('PS_20174392719_1491204439457_log.csv', 'paysim1')
ieee_tx = exactly_one('train_transaction.csv', 'ieee-fraud-detection')
baf = exactly_one('Base.csv', 'bank-account-fraud-dataset-neurips-2022')
ibm = exactly_one('HI-Small_Trans.csv', 'ibm-transactions-for-anti-money-laundering-aml')
print(paysim, ieee_tx, baf, ibm, sep='\n')


In [ ]:
out = Path('/kaggle/working/izqar_unified_mtl_v1')
command = [sys.executable, str(backend / 'scripts' / 'train_unified.py'), '--paysim', str(paysim), '--ieee-dir', str(ieee_tx.parent), '--baf', str(baf), '--ibm-aml', str(ibm), '--output-dir', str(out), '--max-train-rows', '150000', '--max-validation-rows', '75000', '--epochs', '12', '--batch-size', '1024', '--device', 'auto']
subprocess.run(command, cwd=backend, check=True)
print('Validation-only artifacts:', list(out.iterdir()))


Download `unified_weights.npz`, `unified_preprocessors.joblib`, and `unified.metrics.json` only after reviewing the validation comparison. Do not use a previously viewed final test to select architecture, epochs, threshold, or replacement of specialised models.